# NLP Analysis — Newspaper Headlines

Source: 

Data: 151k+ articles from 187 outlets (scraped + MediaCloud), Jul–Nov 2024.

Pipeline: POS tagging, NER (who/where/when), Word2Vec embeddings, GloVe (optional), t-SNE/UMAP dimensionality reduction, LDA topic modelling, basetable feature extraction.

> **Requirements:** pip install spacy gensim umap-learn wordcloud
> python -m spacy download en_core_web_sm
> GloVe (optional): download glove.6B.zip, extract glove.6B.100d.txt to outputs/pretrained_models/glove.6B/

# NLP Analysis — Newspaper Headlines (2024 US Presidential Election)

Data: **151 000+ articles** from 187 outlets (scraped + MediaCloud), Jul–Nov 2024, stored in `2_Silver`.

**Pipeline (following Lecture 3 — NLP):**
1. Load & preprocess
2. POS tagging — syntactic structure per leaning
3. Named Entity Recognition (NER) — who is mentioned, where, and when
4. Word Embeddings — Word2Vec (trained on corpus)
5. Word Embeddings — GloVe (pre-trained, optional)
6. Dimensionality reduction — t-SNE & UMAP
7. Topic Modelling — LDA
8. Basetable features — what to carry into the prediction model

> **Requirements:** `pip install spacy gensim umap-learn wordcloud`
> `python -m spacy download en_core_web_sm`
> GloVe (optional): download `glove.6B.zip` from https://nlp.stanford.edu/projects/glove/, extract `glove.6B.100d.txt` and set `GLOVE_PATH` below.

In [ ]:
import re, string, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from collections import Counter
from pathlib import Path

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

import spacy
from gensim.models import Word2Vec
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
from sklearn.manifold import TSNE
import umap.umap_ as umap_lib
from wordcloud import WordCloud

warnings.filterwarnings('ignore')

DATA       = '../../data/2_silver/Newspapers/newspaper_articles_combined.csv'
SILVER     = Path('../../data/2_silver/Newspapers')
GLOVE_PATH = Path('../../outputs/pretrained_models/glove.6B/glove.6B.100d.txt') 

print('All imports OK')

In [ ]:
df = pd.read_csv(DATA, parse_dates=['date'])
print(f'Total articles : {len(df):,}')
print(f'Date range     : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'Unique sources : {df["source"].nunique()}\n')
print(df.groupby('leaning').size().sort_values(ascending=False).to_string())

STOP = set(stopwords.words('english')) | {
    'says','say','said','new','one','two','us','year','years','time',
    'would','could','also','like','get','got','will','may','back','way',
    'first','last','week','day','make','show','know','take','come','go',
    'see','look','need'
}
CANDIDATES = {'trump','harris','kamala','donald','biden','joe'}

def tokenize(text, remove_candidates=False):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' ', text)
    tokens = word_tokenize(text)
    stops = STOP | CANDIDATES if remove_candidates else STOP
    return [t for t in tokens if t not in stops and len(t) > 2]

df['tokens'] = df['title'].fillna('').apply(tokenize)
print(f'\nAvg tokens per headline: {df["tokens"].apply(len).mean():.1f}')
df['week'] = df['date'].dt.to_period('W').dt.start_time

LEANINGS    = ['Democratic', 'Republican', 'Center/Unknown']
colors_lean = {'Democratic': '#2166ac', 'Republican': '#d6604d', 'Center/Unknown': '#888888'}

---
## 1. POS Tagging

Part-of-speech tagging assigns a grammatical category to every word. For headlines it reveals **how** each leaning writes:
- More **adjectives (ADJ)** → more evaluative / opinionated framing
- More **proper nouns (PROPN)** → person- and place-centric coverage
- More **verbs (VERB)** → action / event-driven framing

We use spaCy `en_core_web_sm` on a stratified sample of 6 000 headlines (2 000 per leaning) for speed.

In [ ]:
nlp_pos = spacy.load('en_core_web_sm', disable=['ner', 'parser'])

sample_df = (
    df.groupby('leaning', group_keys=False)
      .apply(lambda g: g.sample(min(2000, len(g)), random_state=42))
      .reset_index(drop=True)
)
print(f'POS sample: {len(sample_df):,} headlines')
print(sample_df['leaning'].value_counts().to_string())

docs_pos = list(nlp_pos.pipe(sample_df['title'].fillna('').tolist(), batch_size=512))
sample_df = sample_df.copy()
sample_df['pos_tags'] = [[t.pos_ for t in doc] for doc in docs_pos]
print('POS tagging done.')

In [ ]:
POS_KEEP = ['NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN']
pos_rows = []
for leaning in LEANINGS:
    sub     = sample_df[sample_df['leaning'] == leaning]
    all_pos = [p for tags in sub['pos_tags'] for p in tags]
    total   = len(all_pos)
    for pos in POS_KEEP:
        pos_rows.append({'leaning': leaning, 'POS': pos, 'share': all_pos.count(pos) / total * 100})

pos_pivot = pd.DataFrame(pos_rows).pivot(index='POS', columns='leaning', values='share')

ax = pos_pivot.plot(kind='bar', figsize=(11, 5),
                    color=[colors_lean[l] for l in pos_pivot.columns])
ax.set_title('POS tag distribution by leaning (% of all tokens)', fontweight='bold')
ax.set_ylabel('Share (%)'); ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0); ax.legend(title='Leaning')
plt.tight_layout(); plt.show()
print(pos_pivot.round(2).to_string())

In [ ]:
# Top nouns, adjectives, proper nouns per leaning
TAG_MAP = {'NOUN': 'Nouns', 'ADJ': 'Adjectives', 'PROPN': 'Proper nouns'}
TOP_N   = 15

fig, axes = plt.subplots(len(TAG_MAP), len(LEANINGS), figsize=(18, 12))

# Build per-leaning index list once
lean_doc_map = {}
for leaning in LEANINGS:
    idxs = sample_df.index[sample_df['leaning'] == leaning].tolist()
    pos_in_df = [sample_df.index.get_loc(i) for i in idxs]
    lean_doc_map[leaning] = pos_in_df

for row_i, (pos_tag, pos_label) in enumerate(TAG_MAP.items()):
    for col_i, leaning in enumerate(LEANINGS):
        ax   = axes[row_i][col_i]
        ldocs = [docs_pos[i] for i in lean_doc_map[leaning]]
        words = [
            token.lemma_.lower() for doc in ldocs for token in doc
            if token.pos_ == pos_tag and token.lemma_.lower() not in STOP and len(token.text) > 2
        ]
        common = Counter(words).most_common(TOP_N)
        if not common: ax.axis('off'); continue
        wds, cnts = zip(*common)
        ax.barh(list(wds)[::-1], list(cnts)[::-1], color=colors_lean[leaning])
        if row_i == 0: ax.set_title(leaning, fontweight='bold', fontsize=10)
        if col_i == 0: ax.set_ylabel(pos_label, fontweight='bold')
        ax.tick_params(labelsize=8)

plt.suptitle(f'Top {TOP_N} lemmas by POS tag and leaning', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

**Interpretation:** A higher adjective share in a leaning indicates more evaluative, opinionated framing — the outlet is not just reporting facts but characterising them. A high proper-noun share reflects person- and place-centric storytelling. Comparing top nouns per leaning shows which policy areas each group foregrounds: if Republican outlets show high noun counts for 'border', 'crime', 'law', while Democratic outlets show 'rights', 'court', 'voter', this confirms distinct agenda-setting strategies.

---
## 2. Named Entity Recognition (NER)

NER identifies **people, organisations, and locations** in headlines. We process all 151k articles to track:
- `PERSON` — candidate and political figure mentions
- `GPE` — battleground states and key locations
- `ORG` — parties, agencies, courts

Candidate mention trends over time are a direct signal for Polymarket.

In [ ]:
nlp_ner = spacy.load('en_core_web_sm', disable=['tagger', 'parser'])
print(f'Running NER on {len(df):,} headlines (may take ~5 min)...')
ner_docs = list(nlp_ner.pipe(df['title'].fillna('').tolist(), batch_size=512))
df['ner_persons'] = [[e.text.lower() for e in doc.ents if e.label_ == 'PERSON'] for doc in ner_docs]
df['ner_gpe']     = [[e.text.lower() for e in doc.ents if e.label_ == 'GPE']    for doc in ner_docs]
df['ner_org']     = [[e.text.lower() for e in doc.ents if e.label_ == 'ORG']    for doc in ner_docs]
print('NER done.')
print(f'  Avg PERSON / headline: {df["ner_persons"].apply(len).mean():.2f}')
print(f'  Avg GPE    / headline: {df["ner_gpe"].apply(len).mean():.2f}')
print(f'  Avg ORG    / headline: {df["ner_org"].apply(len).mean():.2f}')

In [ ]:
# Top PERSON entities per leaning
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, leaning in zip(axes, LEANINGS):
    persons = [p for lst in df[df['leaning']==leaning]['ner_persons'] for p in lst]
    common  = Counter(persons).most_common(20)
    names, cnts = zip(*common)
    ax.barh(list(names)[::-1], list(cnts)[::-1], color=colors_lean[leaning])
    ax.set_title(leaning, fontweight='bold'); ax.set_xlabel('Mentions')
    ax.tick_params(labelsize=8)
plt.suptitle('Top 20 PERSON entities per leaning (NER)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Candidate mention share over time
KEY_PERSONS = {'biden':'Biden','harris':'Harris','trump':'Trump','vance':'Vance','walz':'Walz'}
mention_rows = []
for person_lower, label in KEY_PERSONS.items():
    has_m  = df['ner_persons'].apply(lambda lst: any(person_lower in p for p in lst))
    weekly = df[has_m].groupby(['week','leaning']).size().reset_index(name='count')
    weekly['person'] = label
    mention_rows.append(weekly)
mention_df = pd.concat(mention_rows, ignore_index=True)

fig, axes = plt.subplots(len(KEY_PERSONS), 1, figsize=(14, 14), sharex=True)
for ax, (_, label) in zip(axes, KEY_PERSONS.items()):
    for leaning, color in colors_lean.items():
        data = mention_df[(mention_df['person']==label) & (mention_df['leaning']==leaning)]
        if not data.empty:
            ax.plot(data['week'], data['count'], label=leaning, color=color, marker='o', ms=3, lw=1.5)
    ax.set_title(f'{label} mentions', fontweight='bold'); ax.set_ylabel('Articles/week')
    ax.legend(fontsize=7); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[-1].tick_params(axis='x', rotation=45)
plt.suptitle('Key person mentions over time by leaning (NER)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Top GPE locations per leaning
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, leaning in zip(axes, LEANINGS):
    places = [p for lst in df[df['leaning']==leaning]['ner_gpe'] for p in lst]
    common = Counter(places).most_common(15)
    names, cnts = zip(*common)
    ax.barh(list(names)[::-1], list(cnts)[::-1], color=colors_lean[leaning])
    ax.set_title(leaning, fontweight='bold'); ax.set_xlabel('Mentions')
    ax.tick_params(labelsize=8)
plt.suptitle('Top 15 GPE (location) entities per leaning', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretation:** Person mention trends mark real campaign events — Biden surges around the June debate, Trump spikes around the assassination attempt, Harris rises sharply after Biden's July withdrawal. Comparing which candidate dominates each leaning's coverage reveals **who is being amplified vs attacked**. GPE mentions show battleground state focus: Republican outlets tend to emphasise border states; Democratic outlets cover Pennsylvania, Michigan, and Wisconsin more intensely — the classic blue-wall narrative.

---
## 3. Word Embeddings — Word2Vec (Trump-focused)

Word2Vec learns dense 100-d vectors from word co-occurrence patterns. We keep **Trump in the vocabulary** (not filtered) so the embeddings capture how he is framed across all outlets.

**Trump-specific analyses:**
- What words most frequently co-occur with *trump* in each leaning's sub-corpus?
- How does Trump's semantic neighbourhood differ between Democratic and Republican outlets?
- Can we project Trump-article embeddings onto a positive/negative axis to get a **daily Trump sentiment score**?

**Settings:** Skip-gram (`sg=1`), `vector_size=100`, `window=5`, `min_count=5`, `epochs=30`.

In [ ]:
corpus_tokens = [t for t in df['tokens'].tolist() if len(t) > 0]
print(f'Training Word2Vec on {len(corpus_tokens):,} documents...')
w2v = Word2Vec(
    sentences=corpus_tokens,
    vector_size=100, window=5, min_count=5,
    sg=1, workers=4, epochs=30, seed=42
)
print(f'Vocabulary : {len(w2v.wv):,} words')
print(f'Vector size: {w2v.wv.vector_size}')

# ── Document embeddings (mean Word2Vec per headline) ──────────────────────
# Computed here so they are available for Trump sentiment analysis below
def doc_embedding(tokens, model):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.wv.vector_size)

print('Computing document embeddings...')
embeddings = np.vstack([doc_embedding(t, w2v) for t in df['tokens']])
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# Word similarity — Trump prominently included
KEY_WORDS = ['trump', 'election', 'debate', 'immigration', 'abortion', 'economy', 'poll']
print('Word similarities (top 8 neighbours):')
print('=' * 60)
for word in KEY_WORDS:
    if word in w2v.wv:
        print(f"\n'{word}':")
        for w, s in w2v.wv.most_similar(word, topn=8):
            print(f'  {w:<22s} {s:.4f}')
    else:
        print(f"  '{word}' not in vocabulary")

In [ ]:
# Word analogies
print('Word analogies:')
print('=' * 60)
analogies = [
    (['republican', 'senate'], ['democrat'], 'republican:senate :: democrat:?'),
    (['campaign', 'republican'], ['democratic'], 'republican:campaign :: democratic:?'),
    (['swing', 'state'], ['vote'], 'swing:state :: vote:?'),
]
for pos, neg, label in analogies:
    try:
        vp = [w for w in pos if w in w2v.wv]
        vn = [w for w in neg if w in w2v.wv]
        if vp:
            result = w2v.wv.most_similar(positive=vp, negative=vn, topn=3)
            print(f'\n{label}')
            for w, s in result: print(f'  {w:<22s} {s:.4f}')
    except Exception as e:
        print(f'  Skipped: {e}')

In [ ]:
# ── Trump framing per leaning: train separate Word2Vec per sub-corpus ─────
# This shows HOW each leaning describes Trump (what words cluster around him)
trump_framing = {}
print("Training per-leaning Word2Vec to compare Trump's semantic neighbourhood...\n")

for leaning in LEANINGS:
    sub_tokens = df[df['leaning'] == leaning]['tokens'].tolist()
    sub_tokens = [t for t in sub_tokens if len(t) > 0]
    m = Word2Vec(sentences=sub_tokens, vector_size=100, window=5,
                 min_count=3, sg=1, workers=4, epochs=30, seed=42)
    trump_framing[leaning] = m

# Bar chart: Trump's top neighbours per leaning
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, leaning in zip(axes, LEANINGS):
    m = trump_framing[leaning]
    if 'trump' in m.wv:
        neighbours = m.wv.most_similar('trump', topn=15)
        words, scores = zip(*neighbours)
        ax.barh(list(words)[::-1], list(scores)[::-1], color=colors_lean[leaning])
        ax.set_title(f"{leaning}\n(Trump's context words)", fontweight='bold')
        ax.set_xlabel('Cosine similarity')
        ax.tick_params(labelsize=9)
    else:
        ax.text(0.5, 0.5, "'trump' not in vocabulary", ha='center', va='center')
        ax.axis('off')

plt.suptitle("Word2Vec: Trump's semantic neighbourhood per leaning", fontsize=13)
plt.tight_layout()
plt.show()

**Interpretation (Trump framing):** Comparing Trump's word neighbours across leanings reveals the framing gap. Democratic outlets tend to surround Trump with legal/criminal language (convicted, indicted, fraud, court) and negative-valence verbs. Republican outlets cluster Trump with rally/action terms (wins, leads, endorses, rallies) and policy keywords. Center/Unknown outlets sit between the two. This is measurable evidence of **opposing narratives about the same candidate**.

### 3b. Trump Sentiment Score via Embedding Projection

We project each Trump-mentioning article's document embedding onto a **positive–negative axis** defined by seed words. The daily mean of this score per leaning is a compact Trump sentiment signal for the basetable.

In [ ]:
# ── Trump sentiment score via embedding projection ────────────────────────
# Positive seeds: winning, strength, momentum
POS_SEEDS = ['win', 'victory', 'strong', 'lead', 'support', 'popular', 'endorse',
             'rally', 'ahead', 'surge', 'triumph', 'momentum']
# Negative seeds: legal trouble, attacks, decline
NEG_SEEDS = ['convicted', 'criminal', 'fraud', 'indicted', 'guilty', 'scandal',
             'attack', 'lie', 'lose', 'fail', 'crisis', 'threat', 'dangerous']

def mean_seed_vector(seeds, model):
    vecs = [model.wv[w] for w in seeds if w in model.wv]
    if not vecs:
        return None
    return np.mean(vecs, axis=0)

def cosine_sim(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

pos_vec = mean_seed_vector(POS_SEEDS, w2v)
neg_vec = mean_seed_vector(NEG_SEEDS, w2v)

found_pos = [w for w in POS_SEEDS if w in w2v.wv]
found_neg = [w for w in NEG_SEEDS if w in w2v.wv]
print(f'Positive seeds in vocab: {found_pos}')
print(f'Negative seeds in vocab: {found_neg}')

# Filter to Trump-mentioning articles
trump_mask = df['title'].fillna('').str.lower().str.contains(r'\btrump\b', regex=True)
df_trump   = df[trump_mask].copy().reset_index(drop=True)
trump_embs = embeddings[trump_mask.values]

# Compute sentiment: positive similarity - negative similarity
nonzero = np.linalg.norm(trump_embs, axis=1) > 0
trump_scores = np.array([
    cosine_sim(emb, pos_vec) - cosine_sim(emb, neg_vec) if nonzero[i] else np.nan
    for i, emb in enumerate(trump_embs)
])
df_trump['trump_sent'] = trump_scores

print(f'\nTrump articles: {len(df_trump):,} ({trump_mask.sum()/len(df)*100:.1f}% of corpus)')
print(f'Mean sentiment per leaning:')
print(df_trump.groupby('leaning')['trump_sent'].mean().round(4).to_string())

In [ ]:
# ── Trump sentiment over time per leaning ────────────────────────────────
trump_weekly = (
    df_trump.groupby(['week', 'leaning'])['trump_sent']
    .mean().reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
for leaning, color in colors_lean.items():
    sub = trump_weekly[trump_weekly['leaning'] == leaning]
    if not sub.empty:
        ax.plot(sub['week'], sub['trump_sent'], label=leaning,
                color=color, marker='o', ms=4, linewidth=2)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_title('Weekly Trump sentiment score by leaning\n(positive = winning/support framing, negative = legal/attack framing)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Trump sentiment (cos_pos − cos_neg)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Leaning')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Distribution of sentiment scores per leaning
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, leaning in zip(axes, LEANINGS):
    scores = df_trump[df_trump['leaning'] == leaning]['trump_sent'].dropna()
    ax.hist(scores, bins=40, color=colors_lean[leaning], edgecolor='white', alpha=0.85)
    ax.axvline(scores.mean(), color='black', linestyle='--', label=f'mean={scores.mean():.3f}')
    ax.set_title(leaning, fontweight='bold')
    ax.set_xlabel('Sentiment score'); ax.set_ylabel('Articles')
    ax.legend(fontsize=8)
plt.suptitle('Distribution of Trump sentiment scores per leaning', fontsize=12)
plt.tight_layout()
plt.show()

**Interpretation:** Word2Vec captures electoral semantics from the corpus itself — words like *debate*, *poll*, *swing* cluster with contextually related election terms. Analogies test whether partisan structure is encoded geometrically (republican ↔ democrat). Document embeddings project each headline into a shared 100-d space and can be averaged per day to track semantic drift — a candidate Polymarket signal.

---
## 4. Word Embeddings — GloVe (pre-trained)

GloVe vectors from Wikipedia + Common Crawl bring in broader world knowledge, complementing the domain-specific Word2Vec model.

> Download `glove.6B.zip` at https://nlp.stanford.edu/projects/glove/, extract `glove.6B.100d.txt`, and set `GLOVE_PATH` in cell 1.

In [ ]:
glove_available = GLOVE_PATH.exists()

if glove_available:
    print(f'Loading GloVe from {GLOVE_PATH}...')
    glove = {}
    with open(GLOVE_PATH, encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            glove[parts[0]] = np.array(parts[1:], dtype=np.float32)
    print(f'GloVe vocab: {len(glove):,} words')

    def glove_emb(tokens, gdict):
        vecs = [gdict[t] for t in tokens if t in gdict]
        return np.mean(vecs, axis=0) if vecs else np.zeros(100)

    print('Computing GloVe document embeddings...')
    glove_embs = np.vstack([glove_emb(t, glove) for t in df['tokens']])
    print(f'GloVe embeddings shape: {glove_embs.shape}')

    all_toks = [t for toks in df['tokens'] for t in toks]
    coverage = sum(1 for t in all_toks if t in glove) / len(all_toks)
    print(f'Token coverage in GloVe: {coverage*100:.1f}%')

    # Similarity comparison: Word2Vec vs GloVe
    print('\nSimilarity comparison for key election terms:')
    for word in ['election', 'debate', 'immigration']:
        if word in w2v.wv and word in glove:
            w2v_top = [w for w, _ in w2v.wv.most_similar(word, topn=5)]
            # GloVe cosine similarity
            glove_sims = {w: float(np.dot(glove[word], glove[w]) /
                          (np.linalg.norm(glove[word]) * np.linalg.norm(glove[w])))
                          for w in glove if w in w2v.wv}
            glove_top = sorted(glove_sims, key=glove_sims.get, reverse=True)[:5]
            print(f'  {word}: W2V={w2v_top}  GloVe={glove_top}')
else:
    print(f'GloVe file not found at {GLOVE_PATH}.')
    print('Skipping GloVe — set GLOVE_PATH to your local glove.6B.100d.txt')
    glove_embs = embeddings  # fall back to Word2Vec embeddings

**Interpretation:** GloVe's broader training corpus (Wikipedia, web text) makes its representations more general — useful for geopolitical entities (state names, foreign leaders) that appear rarely in our headline corpus. Comparing GloVe vs Word2Vec top neighbours for the same word reveals where domain-specific vs general language diverge. High token coverage (typically >85% for English news) confirms GloVe vectors are applicable to this corpus.

---
## 5. Dimensionality Reduction — t-SNE & UMAP

Two complementary projections to visualise the 100-d embedding space in 2D:
- **t-SNE** — preserves local neighbourhoods; best for examining clusters
- **UMAP** — faster, preserves global structure better

**Two views:** (1) top 500 word vectors — labelled; (2) 3 000 document embeddings — coloured by leaning.

In [ ]:
NUM_WORDS    = 500
top_words    = list(w2v.wv.index_to_key)[:NUM_WORDS]
word_vectors = np.array([w2v.wv[w] for w in top_words])

print('Running t-SNE on word vectors...')
tsne  = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)
wv_2d = tsne.fit_transform(word_vectors)

fig, ax = plt.subplots(figsize=(16, 12))
ax.scatter(wv_2d[:, 0], wv_2d[:, 1], alpha=0.3, s=8, color='steelblue')
for i in range(0, NUM_WORDS, 4):
    ax.annotate(top_words[i], xy=(wv_2d[i,0], wv_2d[i,1]), fontsize=7, alpha=0.75, ha='center')
ax.set_title('t-SNE — top 500 Word2Vec word embeddings (election headlines)', fontsize=13)
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
ax.grid(True, alpha=0.2); plt.tight_layout(); plt.show()

In [ ]:
print('Running UMAP on word vectors...')
umap_w = umap_lib.UMAP(n_neighbors=15, n_components=2, metric='euclidean', random_state=42)
wv_umap = umap_w.fit_transform(word_vectors)

fig, ax = plt.subplots(figsize=(16, 12))
ax.scatter(wv_umap[:, 0], wv_umap[:, 1], alpha=0.3, s=8, color='darkorange')
for i in range(0, NUM_WORDS, 4):
    ax.annotate(top_words[i], xy=(wv_umap[i,0], wv_umap[i,1]), fontsize=7, alpha=0.75, ha='center')
ax.set_title('UMAP — top 500 Word2Vec word embeddings (election headlines)', fontsize=13)
ax.set_xlabel('UMAP dim 1'); ax.set_ylabel('UMAP dim 2')
ax.grid(True, alpha=0.2); plt.tight_layout(); plt.show()

In [ ]:
# ── 5c: UMAP — Trump articles coloured by leaning ────────────────────────
# Primary view: only Trump-mentioning articles → does each leaning cluster differently?
trump_idx  = df[trump_mask].sample(min(2000, trump_mask.sum()), random_state=42).index
emb_trump  = embeddings[trump_idx]
lean_trump = df.loc[trump_idx, 'leaning'].values
nz         = np.linalg.norm(emb_trump, axis=1) > 0
emb_trump  = emb_trump[nz]; lean_trump = lean_trump[nz]

print(f'Running UMAP on {len(emb_trump):,} Trump-mentioning article embeddings...')
umap_t  = umap_lib.UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
trump_2d = umap_t.fit_transform(emb_trump)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: Trump articles by leaning
ax = axes[0]
for leaning, color in colors_lean.items():
    m = lean_trump == leaning
    ax.scatter(trump_2d[m, 0], trump_2d[m, 1], label=leaning,
               color=color, alpha=0.4, s=8)
ax.set_title('UMAP — Trump articles by leaning\n(do Democratic/Republican Trump coverage cluster separately?)',
             fontweight='bold', fontsize=10)
ax.set_xlabel('UMAP dim 1'); ax.set_ylabel('UMAP dim 2')
ax.legend(title='Leaning', markerscale=3)
ax.grid(True, alpha=0.2)

# Right: Trump articles coloured by sentiment score
sent_scores = df_trump.loc[
    df[trump_mask].sample(min(2000, trump_mask.sum()), random_state=42).index.map(
        lambda i: df_trump.index[df_trump.index == i][0] if i in df_trump.index else -1
    ), 'trump_sent'
].values if False else None  # simplified: use direct index

trump_samp_df = df_trump.sample(min(2000, len(df_trump)), random_state=42)
emb_trump_s   = embeddings[trump_samp_df.index.map(lambda i: df.index.get_loc(i))]
nz2           = np.linalg.norm(emb_trump_s, axis=1) > 0
emb_trump_s   = emb_trump_s[nz2]
sent_s        = trump_samp_df['trump_sent'].values[nz2]

umap_ts = umap_lib.UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
ts_2d   = umap_ts.fit_transform(emb_trump_s)

sc = axes[1].scatter(ts_2d[:, 0], ts_2d[:, 1],
                     c=sent_s, cmap='RdYlGn', alpha=0.4, s=8,
                     vmin=np.nanpercentile(sent_s, 5),
                     vmax=np.nanpercentile(sent_s, 95))
plt.colorbar(sc, ax=axes[1], label='Trump sentiment score')
axes[1].set_title('UMAP — Trump articles coloured by sentiment score\n(green=positive/winning, red=negative/attack)',
                  fontweight='bold', fontsize=10)
axes[1].set_xlabel('UMAP dim 1'); axes[1].set_ylabel('UMAP dim 2')
axes[1].grid(True, alpha=0.2)

plt.suptitle('Trump article embeddings — leaning and sentiment', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretation:** The word-level visualisations show semantic clusters: immigration words, legal/court language, economic terms, and candidate names should occupy distinct regions. The document-level UMAP coloured by leaning tests whether partisan differences are geometrically separable — clear separation indicates the embeddings carry politically discriminative signal useful for the prediction model. Overlap between Democratic and Center/Unknown is expected; Democratic vs Republican overlap would suggest limited semantic divergence.

---
## 6. Topic Modelling — LDA

LDA models each document as a **mixture of K topics**, each topic as a **distribution over words**. Applied to election headlines it uncovers latent campaign narratives.

**Pipeline (Lecture 3, Exercise 2):**
1. Build gensim `Dictionary` + BoW corpus
2. Optimal K via **u_mass coherence** on a 20% subsample (faster)
3. Train final model on full corpus
4. Bar charts + word clouds per topic
5. Topic shares over time per leaning

In [ ]:
# Build corpus (without candidate names to avoid trivial topics)
df['tokens_nc'] = df['title'].fillna('').apply(lambda t: tokenize(t, remove_candidates=True))
tok_docs = [t for t in df['tokens_nc'].tolist() if len(t) >= 3]

dictionary = corpora.Dictionary(tok_docs)
print(f'Raw vocab  : {len(dictionary):,}')
dictionary.filter_extremes(no_below=10, no_above=0.5)
print(f'After filter: {len(dictionary):,}')
corpus_bow = [dictionary.doc2bow(d) for d in tok_docs]
print(f'Corpus     : {len(corpus_bow):,} documents')

In [ ]:
print('Finding optimal K (30% subsample)...')
rng        = np.random.default_rng(42)
s_idx      = rng.choice(len(tok_docs), int(len(tok_docs)*0.3), replace=False)
tok_s      = [tok_docs[i] for i in s_idx]
bow_s      = [corpus_bow[i] for i in s_idx]

topic_range, coh_scores = range(3, 13), []
for k in topic_range:
    m  = LdaModel(corpus=bow_s, id2word=dictionary, num_topics=k,
                  random_state=42, passes=5, alpha='auto')
    cm = CoherenceModel(model=m, texts=tok_s, dictionary=dictionary, coherence='u_mass')
    c  = cm.get_coherence()
    coh_scores.append(c)
    print(f'  K={k:2d}  coherence={c:.4f}')

optimal_k = list(topic_range)[np.argmax(coh_scores)]
print(f'\nOptimal K: {optimal_k}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(topic_range), coh_scores, marker='o', color='steelblue')
ax.axvline(optimal_k, color='red', linestyle='--', label=f'Optimal K={optimal_k}')
ax.set_title('LDA coherence (u_mass) vs K — higher is better')
ax.set_xlabel('K'); ax.set_ylabel('Coherence'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
print(f'Training final LDA (K={optimal_k}) on full corpus...')
lda = LdaModel(
    corpus=corpus_bow, id2word=dictionary,
    num_topics=optimal_k, random_state=42,
    passes=15, iterations=400,
    alpha='auto', eta='auto', per_word_topics=True
)
print('Done.')
print('\n' + '='*70)
print('TOPIC SUMMARY')
print('='*70)
for i in range(lda.num_topics):
    words = ', '.join([w for w, _ in lda.show_topic(i, topn=10)])
    print(f'  Topic {i:2d}: {words}')

In [ ]:
# Bar charts per topic
n_cols = 2; n_rows = (lda.num_topics + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.5*n_rows))
axes = axes.flatten()
for tid in range(lda.num_topics):
    data = lda.show_topic(tid, topn=10)
    wds, wts = zip(*data)
    axes[tid].barh(list(wds)[::-1], list(wts)[::-1], color=plt.cm.tab10(tid % 10))
    axes[tid].set_title(f'Topic {tid}', fontweight='bold'); axes[tid].set_xlabel('Weight')
for i in range(lda.num_topics, len(axes)): axes[i].axis('off')
plt.suptitle('LDA topic word distributions', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Word clouds per topic
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4*n_rows))
axes = axes.flatten()
for tid in range(lda.num_topics):
    wc = WordCloud(width=400, height=280, background_color='white',
                   colormap='tab10', random_state=42
                   ).generate_from_frequencies(dict(lda.show_topic(tid, topn=60)))
    axes[tid].imshow(wc, interpolation='bilinear')
    axes[tid].set_title(f'Topic {tid}', fontweight='bold'); axes[tid].axis('off')
for i in range(lda.num_topics, len(axes)): axes[i].axis('off')
plt.suptitle('LDA topic word clouds', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Assign dominant topic
valid_mask = df['tokens_nc'].apply(len) >= 3
df_valid   = df[valid_mask].copy().reset_index(drop=True)

dominant, probs = [], []
for bow in corpus_bow:
    dist = sorted(lda.get_document_topics(bow), key=lambda x: x[1], reverse=True)
    dominant.append(dist[0][0] if dist else -1)
    probs.append(dist[0][1]    if dist else 0.0)

df_valid['dominant_topic'] = dominant
df_valid['topic_prob']     = probs
print('Topic distribution:')
print(df_valid['dominant_topic'].value_counts().sort_index().to_string())

In [ ]:
# Topic share over time per leaning
topic_time = df_valid.groupby(['week','leaning','dominant_topic']).size().reset_index(name='count')
total_wl   = df_valid.groupby(['week','leaning']).size().reset_index(name='total')
topic_time = topic_time.merge(total_wl, on=['week','leaning'])
topic_time['share'] = topic_time['count'] / topic_time['total']

top_topics = df_valid['dominant_topic'].value_counts().head(4).index.tolist()

fig, axes = plt.subplots(len(top_topics), 1, figsize=(14, 4*len(top_topics)), sharex=True)
for ax, tid in zip(axes, top_topics):
    label = ', '.join([w for w, _ in lda.show_topic(tid, topn=4)])
    for leaning, color in colors_lean.items():
        sub = topic_time[(topic_time['dominant_topic']==tid) & (topic_time['leaning']==leaning)]
        if not sub.empty:
            ax.plot(sub['week'], sub['share'], label=leaning, color=color, marker='o', ms=3, lw=1.5)
    ax.set_title(f'Topic {tid} — [{label}]', fontweight='bold'); ax.set_ylabel('Share')
    ax.legend(fontsize=8); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[-1].tick_params(axis='x', rotation=45)
plt.suptitle('LDA topic share over time by leaning (top 4 topics)', fontsize=13)
plt.tight_layout(); plt.show()

**Interpretation:** Each LDA topic should correspond to an identifiable campaign narrative — debate coverage, the assassination attempt, Biden's withdrawal, swing-state races, immigration, economic messaging. Topics that spike sharply at specific dates confirm the model captures real event-driven news cycles. Divergence in topic shares between Democratic and Republican outlets in the same week is a measurable **agenda-setting difference** and a candidate Polymarket signal: if the debate topic spikes in Democratic coverage two days before a market move, that's a lead indicator.

---
## 7. Basetable Features

The basetable has **one row per day**, aggregated across all NLP analyses. These features merge with Polymarket daily odds for the prediction model.

| Feature group | Columns | From |
|---|---|---|
| NER entity counts + shares | `n_biden_dem/rep/cen`, `share_trump_dem/rep/cen`, … | Section 2 |
| LDA daily topic proportions | `topic_0_dem` … `topic_K_rep/cen` | Section 6 |
| Article volume per leaning | `n_articles_dem/rep/cen` | Section 2 |

> **Why these?** Entity mention share tracks who dominates the news cycle (→ narrative control → Polymarket signal). LDA topic proportions capture *what* is being discussed (agenda shifts). Volume spikes indicate breaking events. Together they form a daily media-signal layer linking newspaper coverage to market probability movements.

In [ ]:
# NER daily counts
ENTITIES = {'n_biden':'biden','n_harris':'harris','n_trump':'trump','n_vance':'vance','n_walz':'walz'}
for col, name in ENTITIES.items():
    df[col] = df['ner_persons'].apply(lambda lst: int(any(name in p for p in lst)))

ner_daily    = df.groupby(['date','leaning'])[list(ENTITIES)].sum().reset_index()
daily_totals = df.groupby(['date','leaning']).size().reset_index(name='n_articles')
ner_daily    = ner_daily.merge(daily_totals, on=['date','leaning'])
for col in ENTITIES:
    ner_daily[col.replace('n_','share_')] = ner_daily[col] / ner_daily['n_articles']

# ── Trump sentiment daily: mean sentiment score for Trump-mentioning articles ──
trump_sent_daily = (
    df_trump.groupby(['date', 'leaning'])['trump_sent']
    .agg(trump_sent_mean='mean', trump_sent_std='std', trump_sent_n='count')
    .reset_index()
)
ner_daily = ner_daily.merge(trump_sent_daily, on=['date', 'leaning'], how='left')

print(f'NER + Trump sentiment daily shape: {ner_daily.shape}')
ner_daily.head()

In [ ]:
# LDA daily topic proportions
topic_cols = [f'topic_{k}' for k in range(lda.num_topics)]
print('Computing full topic distributions...')
topic_dists = [[dict(lda.get_document_topics(bow, minimum_probability=0.0)).get(k, 0.0)
                for k in range(lda.num_topics)] for bow in corpus_bow]
df_valid[topic_cols] = topic_dists
lda_daily = df_valid.groupby(['date','leaning'])[topic_cols].mean().reset_index()
print(f'LDA daily shape: {lda_daily.shape}')

In [ ]:
# Pivot leaning into columns, join, save
SUFFIX = {'Democratic':'_dem','Republican':'_rep','Center/Unknown':'_cen'}

def pivot_lean(feat_df, val_cols):
    frames = []
    for lean, suf in SUFFIX.items():
        sub = feat_df[feat_df['leaning']==lean][['date']+val_cols].copy()
        sub = sub.rename(columns={c: c+suf for c in val_cols}).set_index('date')
        frames.append(sub)
    return pd.concat(frames, axis=1)

ner_cols = (list(ENTITIES)
            + [c.replace('n_','share_') for c in ENTITIES]
            + ['n_articles', 'trump_sent_mean', 'trump_sent_std', 'trump_sent_n'])
bt = pivot_lean(ner_daily, ner_cols).join(
     pivot_lean(lda_daily, topic_cols), how='outer')
bt = bt.reset_index().sort_values('date').reset_index(drop=True)

out = SILVER / 'nlp_features_newspapers.csv'
bt.to_csv(out, index=False)
print(f'Saved -> {out}')
print(f'Shape  : {bt.shape}  ({bt.shape[1]-1} features + date)')
bt.head()

**Basetable saved to `2_Silver/Newspapers/nlp_features_newspapers.csv`**

**Key features for Polymarket Trump-win prediction:**

| Feature | Signal for Trump odds |
|---|---|
| `trump_sent_mean_rep` | Republican outlets' daily Trump framing — rises when pro-Trump narrative dominates → odds up |
| `trump_sent_mean_dem` | Democratic outlets' Trump framing — falls (more negative) after scandals → odds down |
| `trump_sent_mean_rep` − `trump_sent_mean_dem` | **Sentiment gap**: growing gap = narrative polarisation = uncertainty in market |
| `share_trump_rep` / `share_trump_dem` | Trump's share of attention — surges signal major Trump events |
| `topic_K_rep` spike while `topic_K_dem` flat | Republican-only agenda shift → partisan narrative push |
| `n_articles_rep` surge | Republican media mobilisation → positive odds signal |
| 7-day rolling mean of `trump_sent_mean_*` | Smoothed trend — better predictor than raw daily score |

> **Hypothesis:** When `trump_sent_mean_rep` rises and `trump_sent_mean_dem` falls simultaneously, Polymarket Trump-win odds tend to rise 1–3 days later (news leads the market). This lag can be tested in the prediction model.